# SETUP

In [2]:
#config
import json
from pathlib import Path
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2
import sys

project_root = Path.cwd()
while not (project_root / "src" / "A_Config.py").exists():
    if project_root == project_root.parent:
        raise RuntimeError("couldn't find repo root (src/A_Config.py) above cwd")
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
from A_Config import (
    set_case, case_dir, source_dir, query_dir, sam3_masks_dir,
    frames_for_cam_poses_dir, frames_for_recon_dir, vggt_o_output_dir,
    predictions_path, ply_dir, reg_dir, csv_path_gps, csv_path_pose,
    assets_dir, recon_for_MAP_dir,recon_for_EVENT_dir, agent_p_output_dir,
    set_pass, pass_num
)
from Eval_defs import random_baseline_blocks,random_baseline_closed_form,load_labels,paper_edit_to_timegrid,score_paper_vs_GT_edits,labels_dir,cut_csv_to_timegrid, time_grid,spans_to_mask, annotator_beats_to_grid   # now resolvable
 


case_name =  "503_EGO_noodles_sound" 
experiment = "Pipe_eval_01"
set_case(case_name, experiment)
paper_edit_json_path = agent_p_output_dir() / "503_EGO_noodles_sound_Pipe_eval_01_1_revision.json"
my_cut_csv = labels_dir() / "503_cuts.csv"
BIN_S = 1.0      # vote grid, seconds. 0.1 matches the label precision if you want it finer
WEIGHTS = None   # or {"important": 1.0, "critical": 2.0}


# MY CUT
## VS machine

In [3]:
edges = time_grid(case_name, bin_s=BIN_S, labels=None, tol_s=1.0)
machine_paper_edit = paper_edit_to_timegrid(paper_edit_json_path, case_name, edges)
GT_edit = cut_csv_to_timegrid(my_cut_csv, edges)   # w, 0 or 1
#beta controls f_beta wihc controls soft f1
score_paper_vs_GT_edits(GT_edit, machine_paper_edit)
#f beta is precision and recall together. the F measure
#f-1 is beta =1 same calc but hard coded



503_EGO_noodles_sound: 2/14 beats have no resolved frames — skipped


{'soft_precision': 0.4486111111111111,
 'precision': 0.35833333333333334,
 'soft_recall': 0.35907859078590787,
 'recall': 0.34959349593495936,
 'soft_specificity': 0.8811377245508982,
 'specificity': 0.8616766467065868,
 'iou': 0.215,
 'soft_f1': 0.39888250453196133,
 'f_beta': 0.35390946502057613,
 'beta': 1.0,
 'decay_s': 4.0,
 'true_p_s': 129.0,
 'false_p_s': 231.0,
 'false_n_s': 240.0,
 'true_n_s': 1439.0,
 'edit_s': 360.0,
 'gt_s': 369.0,
 'len_ratio': 0.975609756097561,
 'n_segments': 12,
 'shortest_seg_s': 26.0,
 'median_seg_s': 29.0}

# VS humans 1 min

In [4]:
rows = [dict(editor="ME (GT)", time_min=70.0,
             **score_paper_vs_GT_edits(GT_edit, GT_edit))]

rows.append(dict(editor="machine", time_min=np.nan,
                 **score_paper_vs_GT_edits(GT_edit, machine_paper_edit)))

for l in load_labels():
    rows.append(dict(editor=l["annotator"],
                     time_min=round(l["time_spent_sec"] / 60, 1),
                     **score_paper_vs_GT_edits(GT_edit, annotator_beats_to_grid(l, edges))))
rand_row, null_stats = random_baseline_blocks(GT_edit, machine_paper_edit)
rows.append(rand_row)

rows.append(dict(editor="whole clip", time_min=np.nan,
                 **score_paper_vs_GT_edits(GT_edit, np.ones_like(GT_edit))))


comp = pd.DataFrame(rows)
comp[["editor", "time_min", "soft_f1",
      "soft_specificity", "specificity",
      "soft_precision", "precision",
      "soft_recall", "recall",
      "iou", "len_ratio", "n_segments", "median_seg_s", "edit_s"]].round(3)



,editor,time_min,soft_f1,soft_specificity,specificity,soft_precision,precision,soft_recall,recall,iou,len_ratio,n_segments,median_seg_s,edit_s
0,ME (GT),70.0,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,39.000,8.000,369.0
1,machine,NaN,0.399,0.881,0.862,0.449,0.358,0.359,0.350,0.215,0.976,12.000,29.000,360.0
2,George,84.1,0.397,0.836,0.824,0.361,0.313,0.441,0.363,0.202,1.160,26.000,12.500,428.0
3,Vuk,8.7,0.262,0.849,0.835,0.287,0.220,0.241,0.211,0.121,0.959,14.000,27.000,354.0
4,stefanos,6.7,0.400,0.278,0.222,0.258,0.200,0.892,0.881,0.195,4.401,8.000,181.500,1624.0
5,"random (blocks, n=200)",NaN,0.217,0.836,0.824,0.241,0.185,0.198,0.181,0.101,0.976,11.905,29.005,360.0
6,whole clip,NaN,0.382,0.067,0.000,0.236,0.181,1.000,1.000,0.181,5.526,1.000,2039.000,2039.0


In [5]:
null_stats

,Producer_agent_score,mean,p05,p95,pct
iou,0.215,0.101,0.055,0.155,100.0
soft_f1,0.399,0.217,0.127,0.316,100.0
precision,0.358,0.185,0.106,0.272,100.0
recall,0.350,0.181,0.103,0.266,100.0
specificity,0.862,0.824,0.807,0.843,100.0


In [8]:
cols = ["editor", "time_min", "edit_s", "len_ratio",
        "soft_f1", "soft_precision", "soft_recall", "soft_specificity",
        "iou", "n_segments", "median_seg_s"]

nice = {"editor": "Editor", "time_min": "Time (min)", "edit_s": "Cut (s)",
        "len_ratio": "Len ratio", "soft_f1": "Soft $F_1$",
        "soft_precision": "Soft prec.", "soft_recall": "Soft rec.",
        "soft_specificity": "Soft spec.", "iou": "IoU",
        "n_segments": "Shots", "median_seg_s": "Median shot (s)"}

(comp[cols].rename(columns=nice)
 .to_latex(project_root / "writing" /"Figures" / "sys_eval"/ "tab_noodles_comp.tex",
           index=False, escape=False, float_format="%.3f",
           column_format="lrrrrrrrrrr",
           caption="Editors scored against the reference cut, 503\\_EGO\\_noodles.",
           label="tab:noodles_comp",
           position="htbp"))
